## Install Libraries

In [ ]:
!pip install transformers
!pip install accelerate
!pip install -U bitsandbytes
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.2 MB/s eta 0:00:00:00:0100:01


## Import Libraries

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from transformers import BitsAndBytesConfig
import json
import re

## Load Model for Inference

In [ ]:
### Set Model Name and Huggingface Token Here
model_name = "meta-llama/Llama-3.1-8B"
model_name = "mistralai/Mistral-7B-v0.3"
model_name = "microsoft/phi-2"

access_token = "<HUGGINGFACE ACCESS TOKEN>"


# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)


# Load model and Tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=bnb_config,  trust_remote_code=True, use_auth_token=access_token)
print("model loaded")
print()

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=access_token)
tokenizer.pad_token = tokenizer.eos_token
print("tokenizer loaded")
print()

## Load Dataset

In [ ]:
import json

# load the test/val set of any dataset for K-shot or MMR based K-shot inference
with open("/kaggle/input/sst2-dataset/SST2_Tfidf_topk_test_set.json", "r") as f:
    test_set = json.load(f)
    # test_set = [json.loads(x) for x in f]
    
print(len(test_set))

In [ ]:
test_set[0]

In [ ]:
model

## Zero Shot Inference

In [ ]:
# function to perform zero shot inference
def batch_zero_shot_inference(samples):
    prompts = []
    
    for sample in samples:
        sentence = sample["sentence"]
        
        # Refined prompt for clarity and precision
        prompt = f"""Your task is to judge whether the sentiment of a movie review is positive or negative. 
review: "{sentence}"
sentiment:"""
        
        prompts.append(prompt)

    # Prepare inputs for the model
    model_inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda:0")
    
    # Generate output for all prompts in the batch
    outputs = model.generate(**model_inputs, max_new_tokens=5)  # Greedy decoding for deterministic output
    
    # Decode and return predictions
    predictions = [tokenizer.decode(output, skip_special_tokens=True).strip() for output in outputs]

    return predictions




# set config here
batch_size = 1
k = 0
model_name = 'SET MODEL NAME'
dataset_name = "SET DATASET NAME"


# Inference Loop
with open(f"/kaggle/working/{dataset_name}_test_set_ZS_pred_{model_name}.json", "w") as f:
    
    
    for i in range(0, len(test_set[:]), batch_size):

        # Get the current batch of samples
        batch_samples = test_set[i:i + batch_size]

        predicted_answers = batch_zero_shot_inference(batch_samples)
            

        for j, sample in enumerate(batch_samples):
                predicted_answer = predicted_answers[j]

                answers = re.findall(r'sentiment:\s*(Positive|Negative|positive|negative)', predicted_answer)
                predicted_answer = answers[-1].strip() if len(answers) == (k+1) else None

                if predicted_answer in ['Positive', 'positive']:
                    predicted_answer = 1
                elif predicted_answer in ['Negative', 'negative']:
                    predicted_answer = 0
                else:
                    predicted_answer = None

                temp = {
                    'id': sample['idx'],
                    'sentence': sample['sentence'],
                    'label': sample['label'],
                    'predicted_label': predicted_answer
                }

                if count % 30 == 0:
                    print(count)
                    print(">>>>>>>>>>>>>{}".format(i+1), predicted_answer)
                    print()

                json.dump(temp, f)
                f.write("\n")
                count += 1

## K-shot Inference

In [ ]:
# function to perform few shot inference
def batch_k_shot_inference(samples, k, setting):
    label_map = {1: 'positive', 0: 'negative'}
    prompts  = []
    for sample in samples:
        sentence = sample["sentence"]
        label = sample["label"]

        base = "Your task is to judge whether the sentiment of a movie review is positive or negative. Here are a few examples:\n"

        for i in range(k):
            base += f"""review: {sample[{setting}][i]['sentence']}
sentiment: {label_map[sample[{setting}][i]['label']]} \n"""

        prompt = f"""{base}review: {sentence}\nsentiment:"""

        # print(prompt)
        prompts.append(prompt)
        
    model_inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda:0")
    outputs = model.generate(**model_inputs, max_new_tokens=5)
    predictions = [tokenizer.decode(output, skip_special_tokens=True).strip() for output in outputs]
    return predictions



# set config here
batch_size = 1
model_name = 'SET MODEL NAME'
dataset_name = "SET DATASET NAME"

### Set the config for the experiment you are performing (Few Shot / Few Shot MMR)
setting = 'few_shot'
# setting = 'few_shot_MMR'


# Inference Loop
for k in [1, 3, 5, 7, 9, 10]:
    with open(f"/kaggle/working/{dataset_name}_test_set_pred_k{k}_{model_name}.json", "w") as f:
      
        if setting == 'few_shot':
            setting_value = 'top_k'
        else:
            setting_value = 'reranked_top'
        
        
        for i in range(0, len(test_set[:]), batch_size):

            # Get the current batch of samples
            batch_samples = test_set[i:i + batch_size]

            predicted_answers = batch_k_shot_inference(batch_samples, k, setting=setting_value)
                

            for j, sample in enumerate(batch_samples):
                    predicted_answer = predicted_answers[j]

                    answers = re.findall(r'sentiment:\s*(Positive|Negative|positive|negative)', predicted_answer)
                    predicted_answer = answers[-1].strip() if len(answers) == (k+1) else None

                    if predicted_answer in ['Positive', 'positive']:
                        predicted_answer = 1
                    elif predicted_answer in ['Negative', 'negative']:
                        predicted_answer = 0
                    else:
                        predicted_answer = None

                    temp = {
                        'id': sample['idx'],
                        'sentence': sample['sentence'],
                        'label': sample['label'],
                        'predicted_label': predicted_answer
                    }

                    if count % 30 == 0:
                        print(count)
                        print(">>>>>>>>>>>>>{}".format(i+1), predicted_answer)
                        print()

                    json.dump(temp, f)
                    f.write("\n")
                    count += 1